# Phase 6 — Analyse de Resilience du Reseau Aerien

**Objectif**: Etudier la robustesse du reseau aerien mondial par des metriques de graphe classiques (PAS un modele GNN).

---

## 1. Pourquoi analyser la resilience?

Les phases precedentes ont montre que le reseau aerien est **extremement regulier**: les routes les plus frequentes restent stables dans le temps, et meme un baseline comme HistFreq atteint un MRR de 0.738.

Cette regularite souleve des questions importantes:
- **Quels aeroports sont les plus critiques** pour la connectivite globale?
- **Le reseau est-il fragile** face a la perte de hubs majeurs?
- **Comment le reseau est-il structure** en communautes regionales?

Pour y repondre, nous utilisons des outils d'**analyse de graphe** (centralite, detection de communautes, simulation de perturbation) — et non un modele GNN. Le graphe analyse est construit a partir des interactions de vol reelles du dataset tgbl-flight.

### Distinction interaction vs route

- **Interaction** (dans le dataset): un vol individuel a un instant t — ex: vol Paris→New York le 15 mars 2021
- **Route** (graphe construit ici): une arete ponderee entre deux aeroports, ou le poids = nombre de vols sur cette route

On agrege les interactions en routes pour obtenir un graphe statique analysable par les metriques classiques.

---
## 2. Construction du graphe de routes

On utilise les **3 derniers mois** des donnees d'entrainement (aout-novembre 2021) pour construire un graphe dirige pondere:

- **Noeuds**: aeroports (codes ICAO)
- **Aretes**: routes directes entre aeroports
- **Poids**: nombre de vols sur chaque route pendant la periode

La fenetre de 3 mois evite d'inclure des routes historiques obsoletes tout en capturant suffisamment de structure.

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')

with open('../results/resilience_results.json') as f:
    res = json.load(f)

gs = res['graph_stats']
print('Graphe de routes aeriennes (3 mois: aout-novembre 2021)')
print('=' * 55)
print(f"  Aeroports (noeuds):     {gs['nodes']:>10,}")
print(f"  Routes (aretes):        {gs['edges']:>10,}")
print(f"  Vols totaux (poids):    {gs['total_flights']:>10,}")
print(f"  Vols/route (moyenne):   {gs['total_flights']/gs['edges']:>10.1f}")
print(f"  Routes/aeroport (moy):  {gs['edges']/gs['nodes']:>10.1f}")

Le graphe contient ~14K aeroports et ~505K routes pour ~4.8M vols. En moyenne, chaque route a ~9.6 vols sur 3 mois (environ 1 vol/semaine), et chaque aeroport est connecte a ~36 autres.

La densite est faible: seule une petite fraction des paires d'aeroports possibles ont une route directe. Cela est typique d'un reseau de type **hub-and-spoke** (en etoile).

---
## 3. Metriques de centralite

Trois metriques complementaires pour identifier les aeroports les plus importants:

| Metrique | Mesure | Interpretation |
|----------|--------|----------------|
| **Degre pondere** | Somme des vols (in + out) / max | Volume de trafic brut |
| **Betweenness** | Fraction des plus courts chemins passant par le noeud | Role de "pont" entre regions |
| **PageRank** | Importance recursive (voisins importants → noeud important) | Influence structurelle globale |

In [ ]:
top = res['top_airports']

print('Top 10 aeroports par metrique de centralite')
print('=' * 55)
print(f'{"Rang":<5} {"Degre pondere":<15} {"Betweenness":<15} {"PageRank":<15}')
print('-' * 55)
for i in range(10):
    print(f'{i+1:<5} {top["by_degree"][i]:<15} {top["by_betweenness"][i]:<15} '
          f'{top["by_pagerank"][i]:<15}')

In [ ]:
# Figure 16: Top airports by centrality
top_deg = top['by_degree'][:10]
top_bc = top['by_betweenness'][:10]
top_pr = top['by_pagerank'][:10]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

colors_bar = ['#E91E63'] + ['#64B5F6'] * 9
y_pos = np.arange(10)

for ax, data, title in zip(axes,
                            [top_deg, top_bc, top_pr],
                            ['By Weighted Degree', 'By Betweenness', 'By PageRank']):
    ax.barh(y_pos, range(10, 0, -1), color=colors_bar, edgecolor='white')
    ax.set_yticks(y_pos)
    ax.set_yticklabels(data, fontsize=10)
    ax.set_title(title, fontsize=12)
    ax.invert_yaxis()
    ax.set_xlabel('Rank score', fontsize=10)

plt.suptitle('Top 10 Critical Airports by Centrality Metric', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()
print('Figure 16: Top airports by centrality metric')

### Analyse des resultats de centralite

**KORD (Chicago O'Hare)** domine le degre pondere et le PageRank — c'est le hub avec le plus grand volume de trafic et la plus grande influence structurelle. KATL (Atlanta), KDEN (Denver) et KLAX (Los Angeles) suivent.

**PANC (Anchorage, Alaska)** est le #1 en betweenness, ce qui est remarquable car il n'apparait ni dans le top degre ni dans le top PageRank. Pourquoi?

→ Anchorage est un **pont geographique** entre l'Amerique du Nord et l'Asie-Pacifique. Meme avec un volume modeste, il se trouve sur un grand nombre de plus courts chemins entre ces regions. C'est un point de vulnerabilite unique: sa perte deconnecterait partiellement les reseaux trans-Pacifique.

**KTEB (Teterboro)** en #2 betweenness est aussi surprenant — c'est un aeroport d'aviation privee pres de New York. Sa betweenness elevee vient de son role de pont entre l'aviation generale et les grands hubs du Nord-Est americain.

**Conclusion**: le degre mesure le volume, la betweenness mesure la **criticite structurelle**. Un aeroport a faible volume peut etre vital pour la connectivite.

---
## 4. Detection de communautes

On utilise l'algorithme **Louvain** sur la projection non-dirigee du graphe pour detecter des groupes d'aeroports fortement interconnectes. Ces communautes correspondent souvent a des **regions geographiques** ou des **alliances de compagnies**.

In [ ]:
comm = res['communities']
sizes = comm['sizes']

print(f"Nombre de communautes: {comm['num_communities']}")
print(f"\nTop 10 communautes par taille:")
print(f"{'Rang':<6} {'Taille':<10} {'% du reseau':<12}")
print('-' * 28)
total_shown = 0
for i, s in enumerate(sizes[:10]):
    pct = s / gs['nodes'] * 100
    total_shown += s
    print(f"{i+1:<6} {s:<10} {pct:>8.1f}%")
print(f"\nTop 10 = {total_shown:,} aeroports ({total_shown/gs['nodes']*100:.1f}% du reseau)")
print(f"Restant: {comm['num_communities']-10} communautes, "
      f"{gs['nodes']-total_shown:,} aeroports")

In [ ]:
# Figure 17: Community size distribution
fig, ax = plt.subplots(figsize=(10, 5))
x_comm = range(1, len(sizes) + 1)
ax.bar(x_comm, sizes, color='#4CAF50', edgecolor='white', alpha=0.8)
ax.set_xlabel('Community Rank', fontsize=12)
ax.set_ylabel('Number of Airports', fontsize=12)
ax.set_title(f"Community Size Distribution ({comm['num_communities']} communities)",
             fontsize=13)
ax.set_xlim(0, min(len(sizes) + 1, 25))
plt.tight_layout()
plt.show()
print('Figure 17: Community size distribution')

### Analyse de la structure communautaire

Le reseau se divise en **67 communautes**, avec une distribution tres **asymetrique**:
- La plus grande communaute contient ~2,400 aeroports (~17% du reseau)
- Les 6 plus grandes communautes regroupent ~63% des aeroports
- Les 57 plus petites communautes n'ont que ~37% des aeroports

Cette structure reflete la **hierarchie geographique** de l'aviation mondiale:
- Les grandes communautes correspondent probablement aux marches aeriens majeurs (Amerique du Nord, Europe, Asie)
- Les petites communautes sont des reseaux regionaux ou insulaires (Caraibes, Pacifique Sud, Afrique regionale)

---
## 5. Simulation de perturbation

Que se passe-t-il si des aeroports critiques ferment? On simule deux strategies:

1. **Attaque ciblee**: retrait sequentiel des aeroports par betweenness decroissante (les "ponts" d'abord)
2. **Panne aleatoire**: retrait d'aeroports au hasard

A chaque etape, on mesure:
- La **plus grande composante fortement connexe** (SCC) — le plus grand sous-ensemble ou chaque aeroport peut atteindre tous les autres
- Le **nombre de SCC** — la fragmentation du reseau

In [ ]:
targeted = res['disruption']['targeted']
random_ = res['disruption']['random']

print('Simulation de perturbation (50 aeroports retires)')
print('=' * 60)
print(f"\n{'Etape':<8} {'Ciblee (SCC %)':<18} {'Aleatoire (SCC %)':<18} {'Ecart':<8}")
print('-' * 52)
for step in [0, 10, 20, 30, 40, 50]:
    t = targeted[step]
    r = random_[step]
    diff = (t['connectivity'] - r['connectivity']) * 100
    print(f"{step:<8} {t['connectivity']*100:>12.2f}%     {r['connectivity']*100:>12.2f}%     "
          f"{diff:>+.2f}pp")

drop_t = (targeted[0]['connectivity'] - targeted[50]['connectivity']) * 100
drop_r = (random_[0]['connectivity'] - random_[50]['connectivity']) * 100
print(f"\nBaisse totale: ciblee {drop_t:+.2f}pp, aleatoire {drop_r:+.2f}pp")
print(f"L'attaque ciblee est {drop_t/drop_r:.1f}x plus efficace que la panne aleatoire")

In [ ]:
# Figure 15: Disruption curves
x_t = [r['removed'] for r in targeted]
y_t = [r['connectivity'] * 100 for r in targeted]
x_r = [r['removed'] for r in random_]
y_r = [r['connectivity'] * 100 for r in random_]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(x_t, y_t, 'o-', color='#E91E63', linewidth=2, markersize=3,
         label='Targeted (betweenness)')
ax1.plot(x_r, y_r, 's-', color='#2196F3', linewidth=2, markersize=3,
         label='Random')
ax1.set_xlabel('Airports Removed', fontsize=12)
ax1.set_ylabel('Largest SCC (% of total)', fontsize=12)
ax1.set_title('Network Connectivity Under Disruption', fontsize=13)
ax1.legend(fontsize=11)
ax1.set_ylim(80, 90)

scc_t = [r['num_scc'] for r in targeted]
scc_r = [r['num_scc'] for r in random_]
ax2.plot(x_t, scc_t, 'o-', color='#E91E63', linewidth=2, markersize=3,
         label='Targeted')
ax2.plot(x_r, scc_r, 's-', color='#2196F3', linewidth=2, markersize=3,
         label='Random')
ax2.set_xlabel('Airports Removed', fontsize=12)
ax2.set_ylabel('Number of SCCs', fontsize=12)
ax2.set_title('Network Fragmentation', fontsize=13)
ax2.legend(fontsize=11)

plt.tight_layout()
plt.show()
print('Figure 15: Disruption curves (targeted vs random)')

### Analyse de la resilience

Le resultat principal est frappant: **le reseau aerien est remarquablement resilient**.

Apres avoir retire les 50 aeroports les plus critiques (par betweenness):
- La connectivite ne baisse que de ~85.6% a ~84.1% (−1.5 points de pourcentage)
- Le nombre de SCC augmente de ~2,008 a ~2,169 (+8%)

En comparaison, la panne aleatoire de 50 aeroports ne fait baisser la connectivite que de ~0.3pp.

**Pourquoi le reseau est-il si resilient?**

1. **Haute redondance**: avec ~505K routes entre ~14K aeroports, la suppression d'un noeud redirige le trafic par de nombreux chemins alternatifs

2. **Structure hub-and-spoke distribue**: il n'y a pas UN hub unique mais **des dizaines** de hubs (KORD, KATL, KDEN, KLAX...). La perte d'un seul ne suffit pas a fragmenter le reseau

3. **~85.6% de connectivite initiale**: environ 14.4% des aeroports sont deja dans de petites SCC (petits aeroports regionaux avec peu de connexions). Le "coeur" du reseau est massif et resistant

4. **L'attaque ciblee est ~5x plus efficace** que l'aleatoire, confirmant que le reseau a bien une structure **scale-free** (quelques noeuds sont beaucoup plus importants que la moyenne) — mais meme l'attaque ciblee ne reussit pas a fragmenter significativement le coeur

---
## 6. Synthese: resilience et prediction temporelle

### Lien avec les resultats de prediction

L'analyse de resilience eclaire les resultats des Phases 3-5:

| Observation | Phase resilience | Implication pour la prediction |
|-------------|-----------------|-------------------------------|
| Reseau tres redondant (505K routes) | Le reseau survit a la perte de hubs | Les routes sont stables → HistFreq est fort |
| Structure hub distribue | Pas de point unique de defaillance | La topologie change peu → heuristiques suffisantes |
| PANC comme pont trans-Pacifique | Vulnerabilite geographique specifique | Les routes non-hub sont moins previsibles → TGN les capte mieux (Hits@10) |
| 67 communautes | Marches regionaux distincts | Patterns de vol differents par region → TGN peut apprendre ces nuances |

### Ce que TGN pourrait mieux faire

La resilience montre que le reseau est **localement previsible** (routes stables dans les communautes) mais **globalement complexe** (interactions inter-communautaires, ponts comme PANC). TGN avec un GNN actif pourrait potentiellement exploiter cette structure communautaire pour ameliorer ses predictions sur les routes inter-regionales.

---
## 7. Resume

| Metrique | Valeur |
|----------|--------|
| Aeroports | 13,986 |
| Routes | 505,101 |
| Vols (3 mois) | 4,846,953 |
| Communautes | 67 |
| Plus grande communaute | 2,395 aeroports |
| Connectivite initiale (SCC) | 85.6% |
| Connectivite apres 50 retraits cibles | 84.1% |
| Hub #1 (degre) | KORD (Chicago O'Hare) |
| Pont #1 (betweenness) | PANC (Anchorage) |

**Conclusion principale**: Le reseau aerien mondial est **extremement resilient** grace a sa structure hub-and-spoke distribuee et sa haute redondance. Cette meme regularite structurelle explique pourquoi les heuristiques simples (HistFreq) sont si performantes pour la prediction de liens temporels.

---
## 8. Modules produits

### Code
- `src/graph/resilience.py`: Pipeline complet (construction graphe, centralite, communautes, simulation)
- `scripts/plot_resilience.py`: Generation des figures 15, 16, 17

### Resultats
- `results/resilience_results.json`: Toutes les metriques de resilience
- `results/figures/15_disruption_curves.png`: Courbes de perturbation
- `results/figures/16_top_airports.png`: Top aeroports par centralite
- `results/figures/17_community_sizes.png`: Distribution des tailles de communautes

### Prochaine etape: Phase 7 — Interface Streamlit

Dashboard interactif pour explorer les resultats du projet: predictions TGN, comparaison des modeles, et visualisation de la resilience du reseau.